In [1]:
import math
import os

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tqdm import tqdm

# Gene Exp

In [2]:
gene_exp = pd.concat(
    [
        pd.read_csv("../nci_data/gene_exp_part1.csv.gz", index_col=0).T,
        pd.read_csv("../nci_data/gene_exp_part2.csv.gz", index_col=0).T,
    ],
    axis=1,
)
gene_exp.columns = list(gene_exp.columns)
gene_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
MCF7,1.545084,1.960664,0.045797,0.048305,0.271383,0.000000,0.000000,0.000000,1.790422,0.000000,...,3.251321,4.553231,0.524880,1.325335,2.719968,0.979052,0.955984,2.997876,1.814542,2.017416
MDA_MB_231,0.577656,0.158440,0.023368,0.033706,0.000000,0.020794,0.000000,0.093408,0.281342,0.000000,...,4.150389,4.835798,0.399802,2.148645,1.286207,0.630559,1.366784,3.193204,0.534567,4.290867
HS578T,2.127485,1.114494,0.057868,0.053825,0.000000,0.122192,0.000000,0.000000,0.437279,0.375008,...,1.569525,2.078446,0.211162,0.425895,1.637754,0.183044,1.275240,6.707759,1.183093,1.404440
BT_549,2.807536,1.545173,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.705958,0.000000,...,1.551910,2.247581,0.538375,0.956725,1.921501,0.578255,1.148797,5.211335,1.806192,1.725314
T47D,2.240504,1.404554,0.014713,0.020456,0.083641,0.000000,0.000000,0.205220,0.775457,0.000000,...,2.328319,4.735132,0.634419,1.509372,2.627329,0.850154,1.326106,3.807767,1.399636,1.188945
SF_268,1.904164,0.224674,0.000000,0.066023,0.000000,0.000000,0.000000,0.256671,0.413970,0.126786,...,1.203840,3.447059,0.264858,0.741443,1.187052,0.000000,1.236389,5.422912,0.959441,1.258356
SF_295,1.645900,1.625902,0.021271,0.029497,0.000000,0.000000,0.000000,0.101272,0.493029,0.000000,...,3.338683,4.622304,0.355396,1.124633,2.284713,0.047663,1.388140,7.781976,1.643824,2.399661
SF_539,1.488978,1.449873,0.018674,0.302848,0.426903,0.000000,0.000000,0.116516,0.843719,0.000000,...,2.522377,3.481312,0.798670,1.482724,2.686960,0.119173,1.572594,7.108138,1.645789,2.314656
SNB_19,1.479329,0.274825,0.061902,0.483726,0.152568,0.076873,0.000000,0.145237,2.341513,0.302242,...,1.154635,1.465687,0.813209,1.270068,2.302293,0.121911,1.727816,4.974814,1.340579,1.763526
SNB_75,0.598351,0.248695,0.010085,6.973255,2.752582,0.027438,0.000000,0.167921,2.501286,0.093693,...,2.404099,4.063770,0.576326,1.084949,2.606243,0.095408,1.763505,6.493543,1.689898,2.246688


# Drug Response

In [3]:
drug_response = pd.read_csv("../nci_data/drugAct.csv", index_col=0)
drug_response.columns = gene_exp.index
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,NaN,NaN,NaN,NaN,NaN,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,NaN,NaN,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,NaN,NaN,NaN,NaN,NaN,0.539343,NaN,0.230402,-0.765829,-1.125208,...,NaN,NaN,-0.061612,NaN,NaN,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900911,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,...,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,1.594059,-0.167151,-0.167151,-0.167151
900922,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,...,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786
900964,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,...,-0.158754,NaN,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754
900974,-0.132453,-0.132453,-0.132453,NaN,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,...,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,NaN,-0.132453,-0.132453,-0.132453


In [4]:
nsc_smiles = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
    ["NSC", "SMILES"]
]
nsc_smiles

,NSC,SMILES
0,1,CC1=CC(=O)C=CC1=O
1,17,CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N
2,89,CN(C)CCC(=O)C1=CC=CC=C1.Cl
3,185,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
4,758187,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
...,...,...
23186,799649,C1CC2=C(C1)SC3=NC=NC(=C23)NC4=C(NN=C4)C(=O)NC5...
23187,803209,CCN1C2C(N(C1=O)CC)N3C(=O)C(=C4C5=CC=CC=C5N(C4=...
23188,803778,C1=CC=C(C=C1)C=NC2=C(C(C3=C(O2)C4=CC=CC=C4C=C3...
23189,807047,CC1=C(C=C(C=C1)NC(=O)C2=CC(=NC=C2)C(F)(F)F)C3=...


In [5]:
drug_response = drug_response[drug_response.index.isin(nsc_smiles["NSC"])]
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,NaN,NaN,NaN,NaN,NaN,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,NaN,NaN,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,NaN,NaN,NaN,NaN,NaN,0.539343,NaN,0.230402,-0.765829,-1.125208,...,NaN,NaN,-0.061612,NaN,NaN,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,-0.376238,-0.376238,1.010516,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,...,-0.376238,-0.376238,-0.376238,3.525174,-0.376238,NaN,0.199423,0.443532,-0.376238,-0.376238
831147,1.072608,-0.972979,0.442335,-0.012315,-0.482196,-1.133923,0.744309,0.248854,-0.165824,0.413759,...,-0.351430,-0.514584,-0.611389,0.472510,-0.823481,0.742079,0.422323,-1.046201,-2.375068,-0.500730
831270,1.851876,-0.987088,-0.356117,-0.421592,-0.611842,-0.608036,-0.403236,-0.540247,-0.689865,-0.317744,...,-0.456405,-0.949046,-0.575863,0.841165,1.539347,2.607703,-0.341227,-0.708900,-0.597473,1.611968
831271,0.439331,-0.178969,-0.121072,0.238512,0.339249,-0.395609,-0.142904,-0.070311,-1.578078,-0.067484,...,0.108510,-0.416719,0.151945,0.340800,-0.311599,0.093151,0.349071,-0.108789,-0.327627,0.522628


In [6]:
moa = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
    ["NSC", "MECHANISM"]
]

In [7]:
moa["MECHANISM"].value_counts()

MECHANISM
Other          22245
Kinase           481
DNA              269
TUBB              45
HDAC              45
Apoptosis         36
HSP90             16
Ho                13
Methylation       13
PSM               13
BRD               11
Acetalax           4
Name: count, dtype: int64

In [8]:
not_other = moa[moa["MECHANISM"] != "Other"]
len(not_other)

946

# Also add drugs which are overwrapped with other datasets

In [9]:
tmp = pd.read_csv("../data/drugSynonym.csv")
tmp = tmp[
    (~tmp.nci60.isna() & ~tmp.ctrp.isna())
    | (~tmp.nci60.isna() & ~tmp.gdsc1.isna())
    | (~tmp.nci60.isna() & ~tmp.gdsc2.isna())
]
tmp = [int(i) for i in set(tmp["nci60"].str.split("|").explode())]
tmp

[788121,
 758662,
 766907,
 756652,
 608210,
 765888,
 778909,
 801011,
 761190,
 758706,
 127716,
 761192,
 791226,
 157035,
 613327,
 138783,
 122819,
 77213,
 764040,
 766270,
 773394,
 777425,
 701852,
 750690,
 750691,
 758247,
 111180,
 797938,
 330507,
 87877,
 758871,
 122758,
 760841,
 777639,
 772868,
 727989,
 606698,
 754362,
 753686,
 758872,
 719276,
 757148,
 179940,
 756656,
 759174,
 287459,
 757441,
 90636,
 764036,
 765866,
 178296,
 734950,
 758873,
 174939,
 782007,
 109724,
 759095,
 758664,
 767124,
 800773,
 618487,
 737754,
 766992,
 758612,
 786099,
 755775,
 767600,
 63878,
 764516,
 761431,
 752840,
 339555,
 759878,
 761910,
 765695,
 756642,
 125066,
 758246,
 100880,
 26980,
 764092,
 757306,
 756876,
 758774,
 765776,
 760087,
 766908,
 774829,
 756647,
 800085,
 759155,
 279836,
 756645,
 781516,
 319726,
 757036,
 727125,
 764239,
 800955,
 778368,
 759659,
 755806,
 777718,
 721517,
 760843,
 755606,
 698037,
 755386,
 656576,
 758645,
 765694,
 70699

# To fit into the memory for A100 80GB, we select MoA which is not "Other"

In [10]:
len(set(drug_response.index) & (set(list(not_other["NSC"])) | set(tmp)))

1005

In [11]:
drug_response = drug_response.loc[
    sorted(set(drug_response.index) & (set(list(not_other["NSC"])) | set(tmp)))
]
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
740,0.703626,-1.219032,-1.892792,-0.877267,-1.156158,0.510978,0.536910,0.589970,-0.356387,-1.460314,...,0.626838,0.507493,0.682664,-0.891882,0.499337,0.541612,-1.400771,0.602244,-1.641942,0.231533
752,0.475296,-0.312852,-1.089067,-0.441030,-0.058619,0.057507,0.125700,0.111693,-3.285729,-0.114051,...,-0.321691,0.507798,0.384102,-1.314527,-0.318444,0.557175,0.345056,-0.047731,0.155244,-0.160223
755,0.704027,-0.438857,-0.548744,-1.441942,0.496864,0.096265,-0.082186,0.417634,-1.927502,-0.372021,...,0.123647,0.543639,0.623318,-1.374212,-0.173024,0.314436,-1.002183,-0.881252,0.491364,-0.183200
757,0.318209,NaN,NaN,NaN,NaN,0.307045,0.355509,0.385020,0.183820,-0.384083,...,NaN,NaN,0.009710,0.225675,-0.597319,-0.093596,0.357496,0.158630,-3.402035,-1.148675
762,0.547964,-1.033803,-1.399273,-0.538268,1.137432,0.135942,-0.094460,0.562628,-1.398911,-1.050409,...,-0.294845,0.934592,0.591263,-0.552673,1.797227,1.260987,0.172806,0.869675,-0.529810,-0.634413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811429,0.088052,0.332208,1.572703,0.816968,-0.194672,0.607785,0.306740,1.381493,-0.240118,1.766007,...,-0.621433,-0.040864,0.201668,1.999485,0.432290,0.532053,1.847915,0.879530,-0.113922,1.113174
812926,-0.692843,-0.198620,-0.091472,-0.363542,0.716930,0.114785,-0.743516,-0.403585,-0.959310,0.424058,...,-0.737629,-0.105010,0.195539,1.856611,2.094631,0.843571,0.602042,-0.385549,3.111755,1.518265
812927,3.663091,0.176469,-0.137450,-0.461071,0.029122,-0.540453,-0.393660,0.141554,-0.019758,2.130226,...,-0.553986,-0.602355,-0.438735,0.000707,-0.047693,-1.150848,0.071027,-0.839514,0.103856,0.116587
813488,0.319494,0.259510,0.959946,1.353693,0.058727,0.236707,-0.821851,0.621550,-0.251549,2.256322,...,-0.563012,-0.240817,0.742293,5.112669,0.323573,-0.185610,1.349165,0.191249,0.266489,1.166435


In [12]:
df = pd.DataFrame()
for i in drug_response.columns:
    tmp = pd.DataFrame(drug_response[i]).reset_index().dropna()
    tmp["cell"] = [i] * len(tmp)
    tmp.columns = ["Drug", "Value", "Cell"]
    tmp = tmp[["Drug", "Cell", "Value"]]
    df = pd.concat([df, tmp])
df

,Drug,Cell,Value
0,740,MCF7,0.703626
1,752,MCF7,0.475296
2,755,MCF7,0.704027
3,757,MCF7,0.318209
4,762,MCF7,0.547964
...,...,...,...
1000,811429,UO_31,1.113174
1001,812926,UO_31,1.518265
1002,812927,UO_31,0.116587
1003,813488,UO_31,1.166435


# Zero shot prediction

In [13]:
unique_drugs = df["Drug"].unique()
unique_cells = df["Cell"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[(df["Drug"].isin(train_drugs)) & (df["Cell"].isin(train_cells))]
val_df = df[(df["Drug"].isin(val_drugs)) & (df["Cell"].isin(val_cells))]
test_df = df[(df["Drug"].isin(test_drugs)) & (df["Cell"].isin(test_cells))]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Value"] > 0]
    negative = df[df["Value"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["Drug", "Cell"]]
y_train = np.array(train_df["Value"] > 0, dtype=float)

X_val = val_df[["Drug", "Cell"]]
y_val = np.array(val_df["Value"] > 0, dtype=float)

X_test = test_df[["Drug", "Cell"]]
y_test = np.array(test_df["Value"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

X_train.to_csv("../nci_data/train.csv", index=False)
X_test.to_csv("../nci_data/test.csv", index=False)
X_val.to_csv("../nci_data/val.csv", index=False)

np.save("../nci_data/train_labels.npy", y_train)
np.save("../nci_data/test_labels.npy", y_test)
np.save("../nci_data/val_labels.npy", y_val)

Train:
(12398, 2) (12398,)
Percentage: 62.93%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(3186, 2) (3186,)
Percentage: 16.17%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(4116, 2) (4116,)
Percentage: 20.89%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 19700
Ratio (Train:Validation:Test): 12398:3186:4116
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [14]:
drug_response = drug_response.fillna(0)
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
740,0.703626,-1.219032,-1.892792,-0.877267,-1.156158,0.510978,0.536910,0.589970,-0.356387,-1.460314,...,0.626838,0.507493,0.682664,-0.891882,0.499337,0.541612,-1.400771,0.602244,-1.641942,0.231533
752,0.475296,-0.312852,-1.089067,-0.441030,-0.058619,0.057507,0.125700,0.111693,-3.285729,-0.114051,...,-0.321691,0.507798,0.384102,-1.314527,-0.318444,0.557175,0.345056,-0.047731,0.155244,-0.160223
755,0.704027,-0.438857,-0.548744,-1.441942,0.496864,0.096265,-0.082186,0.417634,-1.927502,-0.372021,...,0.123647,0.543639,0.623318,-1.374212,-0.173024,0.314436,-1.002183,-0.881252,0.491364,-0.183200
757,0.318209,0.000000,0.000000,0.000000,0.000000,0.307045,0.355509,0.385020,0.183820,-0.384083,...,0.000000,0.000000,0.009710,0.225675,-0.597319,-0.093596,0.357496,0.158630,-3.402035,-1.148675
762,0.547964,-1.033803,-1.399273,-0.538268,1.137432,0.135942,-0.094460,0.562628,-1.398911,-1.050409,...,-0.294845,0.934592,0.591263,-0.552673,1.797227,1.260987,0.172806,0.869675,-0.529810,-0.634413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811429,0.088052,0.332208,1.572703,0.816968,-0.194672,0.607785,0.306740,1.381493,-0.240118,1.766007,...,-0.621433,-0.040864,0.201668,1.999485,0.432290,0.532053,1.847915,0.879530,-0.113922,1.113174
812926,-0.692843,-0.198620,-0.091472,-0.363542,0.716930,0.114785,-0.743516,-0.403585,-0.959310,0.424058,...,-0.737629,-0.105010,0.195539,1.856611,2.094631,0.843571,0.602042,-0.385549,3.111755,1.518265
812927,3.663091,0.176469,-0.137450,-0.461071,0.029122,-0.540453,-0.393660,0.141554,-0.019758,2.130226,...,-0.553986,-0.602355,-0.438735,0.000707,-0.047693,-1.150848,0.071027,-0.839514,0.103856,0.116587
813488,0.319494,0.259510,0.959946,1.353693,0.058727,0.236707,-0.821851,0.621550,-0.251549,2.256322,...,-0.563012,-0.240817,0.742293,5.112669,0.323573,-0.185610,1.349165,0.191249,0.266489,1.166435


# Masking

In [15]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# DTI

In [16]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="NSC").reset_index(drop=True)
dti["NSC"] = dti["NSC"].astype(int)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,APOD,122759
1,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,PTGDS,122759
2,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH12,122759
3,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH5,122759
4,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH13,122759


In [17]:
dti = dti[dti["NSC"].isin(drug_response.index)]
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
36,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239
37,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239
38,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,3061
39,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,757306
40,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,3061


## All drugs are in drug response.

In [18]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
36,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239
37,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239
38,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,3061
39,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,DHFR,757306
40,Pyrimethamine,DB00205,4993.0,46505987.0,CCC1=C(C(N)=NC(N)=N1)C1=CC=C(Cl)C=C1,HEXB,3061
...,...,...,...,...,...,...,...
1496,Futibatinib,DB15149,NaN,NaN,COC1=CC(=CC(OC)=C1)C#CC1=NN([C@H]2CCN(C2)C(=O)...,FGFR2,813488
1497,Futibatinib,DB15149,NaN,NaN,COC1=CC(=CC(OC)=C1)C#CC1=NN([C@H]2CCN(C2)C(=O)...,FGFR3,813488
1498,Futibatinib,DB15149,NaN,NaN,COC1=CC(=CC(OC)=C1)C#CC1=NN([C@H]2CCN(C2)C(=O)...,FGFR4,813488
1500,BMS-754807,DB15399,NaN,NaN,C[C@]1(CCCN1C1=NN2C=CC=C2C(NC2=NNC(=C2)C2CC2)=...,IGF1R,758243


In [19]:
print("unique drugs:", len(set(dti.NSC)))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 200
unique genes: 251


# Selected highly variable genes

In [20]:
print(
    "Density: ",
    round((len(gene_exp.values.nonzero()[0]) / gene_exp.size) * 100, 2),
    "%",
)

Density:  75.09 %


In [21]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0]].index)
len(variance)

2383

In [22]:
gene_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
MCF7,1.545084,1.960664,0.045797,0.048305,0.271383,0.000000,0.000000,0.000000,1.790422,0.000000,...,3.251321,4.553231,0.524880,1.325335,2.719968,0.979052,0.955984,2.997876,1.814542,2.017416
MDA_MB_231,0.577656,0.158440,0.023368,0.033706,0.000000,0.020794,0.000000,0.093408,0.281342,0.000000,...,4.150389,4.835798,0.399802,2.148645,1.286207,0.630559,1.366784,3.193204,0.534567,4.290867
HS578T,2.127485,1.114494,0.057868,0.053825,0.000000,0.122192,0.000000,0.000000,0.437279,0.375008,...,1.569525,2.078446,0.211162,0.425895,1.637754,0.183044,1.275240,6.707759,1.183093,1.404440
BT_549,2.807536,1.545173,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.705958,0.000000,...,1.551910,2.247581,0.538375,0.956725,1.921501,0.578255,1.148797,5.211335,1.806192,1.725314
T47D,2.240504,1.404554,0.014713,0.020456,0.083641,0.000000,0.000000,0.205220,0.775457,0.000000,...,2.328319,4.735132,0.634419,1.509372,2.627329,0.850154,1.326106,3.807767,1.399636,1.188945
SF_268,1.904164,0.224674,0.000000,0.066023,0.000000,0.000000,0.000000,0.256671,0.413970,0.126786,...,1.203840,3.447059,0.264858,0.741443,1.187052,0.000000,1.236389,5.422912,0.959441,1.258356
SF_295,1.645900,1.625902,0.021271,0.029497,0.000000,0.000000,0.000000,0.101272,0.493029,0.000000,...,3.338683,4.622304,0.355396,1.124633,2.284713,0.047663,1.388140,7.781976,1.643824,2.399661
SF_539,1.488978,1.449873,0.018674,0.302848,0.426903,0.000000,0.000000,0.116516,0.843719,0.000000,...,2.522377,3.481312,0.798670,1.482724,2.686960,0.119173,1.572594,7.108138,1.645789,2.314656
SNB_19,1.479329,0.274825,0.061902,0.483726,0.152568,0.076873,0.000000,0.145237,2.341513,0.302242,...,1.154635,1.465687,0.813209,1.270068,2.302293,0.121911,1.727816,4.974814,1.340579,1.763526
SNB_75,0.598351,0.248695,0.010085,6.973255,2.752582,0.027438,0.000000,0.167921,2.501286,0.093693,...,2.404099,4.063770,0.576326,1.084949,2.606243,0.095408,1.763505,6.493543,1.689898,2.246688


In [23]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582


# Preprocessed data dims

In [24]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(60, 2582)

In [25]:
drug_response.shape

(1005, 60)

# Normalize

In [26]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,-0.376324,2.310284,-0.370815,-0.397579,-0.579935,-0.932530,1.106057,0.387136,1.421020,-0.736827,...,-0.285390,-0.715825,-0.917886,-2.518276,-0.779108,0.932905,0.615674,1.805885,-0.815677,-1.575934
MDA_MB_231,-0.387264,-1.011604,-0.431717,-0.397579,-0.934955,-0.243114,0.676143,-1.412604,1.663180,-0.716900,...,-1.838190,-0.228654,0.227394,1.423209,-0.779108,-0.855846,1.578712,-0.832061,1.862555,-1.454818
HS578T,-0.372187,-0.980388,-0.325888,-0.225341,-1.300565,-0.591523,-0.370487,-0.159464,-0.827946,0.028936,...,0.802006,-1.185070,-0.243143,-0.324218,-0.114256,0.904508,-1.160363,-0.287149,-0.336022,0.724444
BT_549,-0.412522,1.331116,-0.431717,-0.397579,-1.021722,-0.142835,-1.038877,0.432084,-1.092816,0.217943,...,1.320986,-0.119063,-0.023918,-0.986399,1.449155,-0.673092,-1.450610,1.423024,2.390315,-0.203440
T47D,-0.397193,1.878022,-0.416104,-0.397579,-1.257820,-0.091678,2.024752,-0.322621,-0.186534,-0.118696,...,-0.332664,0.400231,0.451186,0.404201,1.552933,1.229210,1.145791,0.907319,1.683829,-1.073747
SF_268,-0.363046,0.115596,-0.431717,-0.397579,-1.719511,-1.131959,-1.968854,-0.957320,-0.854023,-0.862269,...,0.438404,-0.698035,0.812874,-0.098882,1.110562,-0.351052,-0.947517,-1.084631,-0.700465,-0.072248
SF_295,-0.390418,-0.249157,0.686757,-0.337212,1.219555,1.348892,-0.603617,0.719695,-0.192467,1.009630,...,1.055815,0.292586,-0.003184,0.989731,1.033240,0.322160,0.771929,2.211958,-0.882771,1.390531
SF_539,-0.185578,-1.011043,-0.431717,-0.365347,0.963972,0.293545,-0.435006,1.305936,0.220623,1.781673,...,0.520796,0.151541,-0.239729,-0.093653,0.197136,1.737417,0.314128,0.218988,-0.544703,0.972705
SNB_19,-0.050035,0.419132,-0.369670,-0.297337,-0.980557,1.303514,-0.520841,0.581188,0.084009,0.440593,...,2.691975,1.428340,-0.207826,0.057873,1.589442,0.204382,-1.600902,-1.091802,-0.757313,-0.350099
SNB_75,4.812989,-0.390628,-0.411499,-0.210385,0.746291,0.227105,0.296437,0.528289,-0.825563,1.304365,...,1.572359,1.134967,0.427799,-0.089610,1.824346,0.272590,-0.699092,-0.740624,1.296633,0.591615


In [27]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,-0.983581,0.567781,-0.976554,-1.004311,-0.030365,-0.867297,0.137685,-0.062226,0.539043,-0.213555,...,0.068459,-0.402870,-1.004311,-1.004311,-1.004311,0.542967,0.475431,0.731664,-0.928765,0.282236
MDA_MB_231,-0.898728,-0.871638,-0.911263,-0.911263,-0.150181,-0.382089,-0.034996,-0.853015,0.518682,-0.217658,...,-0.692272,-0.196866,-0.423512,0.764109,-0.911263,-0.428974,0.738848,-0.558363,0.297709,0.276238
HS578T,-1.124311,-1.083195,-1.096359,-1.059812,-0.282807,-0.737077,-0.379520,-0.397794,-0.536672,0.142348,...,0.681687,-0.719975,-0.781212,0.046884,-0.749300,0.553251,-0.374961,-0.391857,-0.803155,2.051378
BT_549,-1.140064,0.029856,-1.140064,-1.140064,-0.248784,-0.431592,-0.629763,-0.135119,-0.686885,0.166994,...,0.862791,-0.228284,-0.682019,-0.356678,0.118886,-0.454298,-0.546880,0.468719,0.585543,1.191549
T47D,-1.095793,0.218823,-1.097394,-1.104249,-0.341995,-0.415769,0.265066,-0.529092,-0.300487,-0.053781,...,-0.094894,-0.032820,-0.456091,0.276679,0.113188,0.544340,0.546281,0.132026,0.154877,0.469901
SF_268,-0.983777,-0.424864,-1.013360,-1.013360,-0.317088,-1.013360,-0.797695,-0.712112,-0.449568,-0.251269,...,0.505764,-0.376918,-0.125277,0.225690,0.055839,-0.140374,-0.187793,-0.720996,-0.875217,1.416440
SF_295,-1.246225,-0.886228,-0.779467,-1.231423,0.111648,0.342261,-0.674187,-0.221748,-0.477125,0.279519,...,0.414237,-0.260027,-0.836012,0.357588,-0.335897,-0.122908,0.195964,0.563939,-1.218216,1.877669
SF_539,-1.078624,-1.163949,-1.210668,-1.195250,0.201487,-0.215655,-0.526731,0.200238,-0.180523,0.833436,...,0.311800,-0.196296,-0.872054,-0.002342,-0.673155,0.814035,0.157676,-0.259047,-0.998269,1.888544
SNB_19,-0.909958,-0.359886,-1.109004,-1.088003,-0.179095,0.716989,-0.422116,0.010074,-0.081747,0.371134,...,1.795855,0.616581,-0.753023,0.262318,0.284461,0.129253,-0.582339,-0.833562,-1.024613,1.229507
SNB_75,1.775623,-0.964880,-1.296272,-1.215007,0.065035,-0.344336,-0.383487,-0.265073,-0.736815,0.527585,...,0.809050,0.185738,-0.624784,-0.079106,0.147003,-0.089147,-0.378824,-0.839009,-0.156327,1.563645


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [28]:
def min_max_scale(data):
    data = data[data > 0].fillna(0)
    scaler = MinMaxScaler(feature_range=(0, 1))
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [29]:
A_dc = min_max_scale(drug_response)
A_dc

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
740,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.159753,0.000000,0.000000,...,0.0,0.162567,0.109683,0.000000,0.000000,0.089914,0.000000,0.000000,0.000000,0.031215
752,0.081408,0.000000,0.000000,0.000000,0.000000,0.024577,0.017263,0.030245,0.000000,0.000000,...,0.0,0.162665,0.061714,0.000000,0.000000,0.092498,0.049962,0.000000,0.039326,0.000000
755,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.113088,0.000000,0.000000,...,0.0,0.174146,0.100148,0.000000,0.000000,0.052200,0.000000,0.000000,0.124472,0.000000
757,0.054502,0.000000,0.000000,0.000000,0.000000,0.131221,0.048824,0.104256,0.130015,0.000000,...,0.0,0.000000,0.000000,0.038995,0.000000,0.000000,0.000000,0.032982,0.000000,0.000000
762,0.093854,0.000000,0.000000,0.000000,0.313990,0.058097,0.000000,0.152350,0.000000,0.000000,...,0.0,0.299381,0.094998,0.000000,0.407777,0.209339,0.025021,0.180820,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811429,0.015081,0.102516,0.394201,0.204184,0.000000,0.259748,0.042127,0.374084,0.000000,0.268540,...,0.0,0.000000,0.032402,0.345499,0.098083,0.088327,0.267569,0.182869,0.000000,0.150076
812926,0.000000,0.000000,0.000000,0.000000,0.197910,0.049056,0.000000,0.000000,0.000000,0.064482,...,0.0,0.000000,0.031417,0.320812,0.475256,0.140043,0.087173,0.000000,0.788265,0.204690
812927,0.627408,0.054456,0.000000,0.000000,0.008039,0.000000,0.000000,0.038330,0.000000,0.323923,...,0.0,0.000000,0.000000,0.000122,0.000000,0.000000,0.010284,0.000000,0.026309,0.015718
813488,0.054722,0.080082,0.240613,0.338327,0.016212,0.101161,0.000000,0.168305,0.000000,0.343097,...,0.0,0.000000,0.119264,0.883439,0.073416,0.000000,0.195352,0.039764,0.067507,0.157257


In [30]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.370515,0.086808,0.512226,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.465915,0.228217,0.762567,0.000000,0.000000
MDA_MB_231,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.191000,0.000000,0.570189,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.484745,0.000000,0.725930,0.000000
HS578T,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.061157,...,0.330604,0.000000,0.000000,0.000000,0.000000,0.460197,0.000000,0.000000,0.000000,0.718860
BT_549,0.000000,0.472878,0.000000,0.000000,0.000000,0.000000,0.000000,0.079342,0.000000,0.137443,...,0.486600,0.000000,0.000000,0.000000,0.368038,0.000000,0.000000,0.568494,1.000000,0.255892
T47D,0.000000,0.728561,0.000000,0.000000,0.000000,0.000000,0.682145,0.000000,0.000000,0.000000,...,0.000000,0.179670,0.000000,0.311285,0.391058,0.559888,0.353917,0.312337,0.617874,0.000000
SF_268,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.210384,0.000000,0.154314,0.057974,0.273768,0.000000,0.000000,0.000000,0.000000,0.348108
SF_295,0.000000,0.000000,0.000000,0.000000,0.603709,0.549280,0.000000,0.133039,0.000000,0.460296,...,0.327564,0.015921,0.000000,0.615968,0.163675,0.062901,0.202446,0.834194,0.000000,0.846372
SF_539,0.000000,0.000000,0.000000,0.000000,0.528543,0.025298,0.000000,0.402412,0.010479,0.933735,...,0.185523,0.000000,0.000000,0.000000,0.000000,0.805463,0.098683,0.000000,0.000000,0.740983
SNB_19,0.000000,0.020586,0.000000,0.000000,0.000000,0.656252,0.000000,0.157970,0.000591,0.289830,...,1.000000,1.000000,0.000000,0.146385,0.439827,0.105325,0.000000,0.000000,0.000000,0.227742
SNB_75,1.000000,0.000000,0.000000,0.000000,0.367942,0.000000,0.000000,0.070325,0.000000,0.654105,...,0.530637,0.645847,0.000000,0.000000,0.462699,0.057911,0.000000,0.000000,0.383186,0.558152


In [31]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[int(i["NSC"]), i["Gene"]] = 1
A_dg

,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
740,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
752,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
755,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
757,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
762,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811429,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
812926,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
812927,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
813488,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [32]:
A_dg.sum().sum()

1297758.0

In [33]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  38.62 %
Cell Density:  45.540000000000006 %
Gene Density:  100.0 %


# Similarity

In [34]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [35]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../nci_data/cell_sim.csv")
cell_sim

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
MCF7,1.000000,0.091148,0.097479,0.189910,0.264853,0.260940,0.308346,0.301156,0.168405,0.143665,...,0.275445,0.235828,0.274520,0.090947,0.243209,0.148226,0.261767,0.279787,0.098606,0.144671
MDA_MB_231,0.091148,1.000000,0.236713,0.299046,0.153116,0.282035,0.147971,0.149372,0.258491,0.061478,...,0.347378,0.210362,0.229396,0.049011,0.160505,0.097864,0.205049,0.266621,0.249394,0.121447
HS578T,0.097479,0.236713,1.000000,0.255637,0.142899,0.171487,0.134565,0.153475,0.127367,0.153692,...,0.272006,0.092625,0.166615,0.115165,0.097272,0.075460,0.244480,0.160875,0.178296,0.098436
BT_549,0.189910,0.299046,0.255637,1.000000,0.303038,0.439894,0.278444,0.348725,0.336355,0.143203,...,0.439289,0.309444,0.421629,0.112152,0.244465,0.180701,0.335153,0.389244,0.198106,0.215764
T47D,0.264853,0.153116,0.142899,0.303038,1.000000,0.320490,0.264082,0.229959,0.239270,0.122100,...,0.361556,0.261223,0.293035,0.085810,0.251317,0.181088,0.244079,0.323435,0.178821,0.235660
SF_268,0.260940,0.282035,0.171487,0.439894,0.320490,1.000000,0.399016,0.449761,0.574509,0.145265,...,0.478146,0.598044,0.581712,0.084511,0.440453,0.274908,0.337637,0.663886,0.194951,0.232815
SF_295,0.308346,0.147971,0.134565,0.278444,0.264082,0.399016,1.000000,0.374788,0.268687,0.145373,...,0.352885,0.306691,0.360767,0.096910,0.304306,0.205345,0.300473,0.381351,0.120485,0.181005
SF_539,0.301156,0.149372,0.153475,0.348725,0.229959,0.449761,0.374788,1.000000,0.287546,0.204457,...,0.317757,0.347749,0.478665,0.113540,0.328980,0.198982,0.374695,0.427560,0.101011,0.166793
SNB_19,0.168405,0.258491,0.127367,0.336355,0.239270,0.574509,0.268687,0.287546,1.000000,0.080607,...,0.389389,0.461809,0.407669,0.042424,0.279377,0.157328,0.208455,0.482008,0.156032,0.149653
SNB_75,0.143665,0.061478,0.153692,0.143203,0.122100,0.145265,0.145373,0.204457,0.080607,1.000000,...,0.155035,0.085208,0.149449,0.136039,0.120567,0.090138,0.295595,0.140737,0.063685,0.114384


In [36]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 0.9999999999999999
Mean: 0.2599368405695073
Median: 0.2397313158545162


In [37]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
os.makedirs("../nci_data/gene_sim", exist_ok=True)
chunks = np.array_split(gene_sim, 25)

for i, chunk in tqdm(enumerate(chunks)):
    chunk.to_parquet(
        f"../nci_data/gene_sim/gene_sim_part_{i}.parquet", compression="gzip"
    )
gene_sim

/Users/inouey2/base/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
25it [00:06,  4.11it/s]


,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
A2M,1.000000,0.060190,0.090301,0.391793,0.099959,0.062216,0.166070,0.340668,0.138102,0.156964,...,0.185882,0.210333,0.128977,0.165382,0.119324,0.222604,0.069420,0.076629,0.097455,0.140388
ABCA3,0.060190,1.000000,0.275980,0.048084,0.215076,0.174623,0.159356,0.058527,0.087342,0.135836,...,0.194404,0.154588,0.187560,0.078862,0.244293,0.101630,0.123893,0.361435,0.203343,0.167400
ABCB1,0.090301,0.275980,1.000000,0.087243,0.240955,0.132595,0.166354,0.080465,0.104796,0.189396,...,0.176134,0.141762,0.229688,0.131027,0.180989,0.128992,0.187949,0.200042,0.174078,0.223905
ABCB5,0.391793,0.048084,0.087243,1.000000,0.063024,0.041796,0.164287,0.373311,0.186304,0.119026,...,0.162311,0.170567,0.114037,0.193312,0.068295,0.224850,0.086748,0.064751,0.059511,0.115838
ABCC1,0.099959,0.215076,0.240955,0.063024,1.000000,0.349966,0.127889,0.114741,0.122463,0.340527,...,0.202791,0.227102,0.182738,0.139628,0.134596,0.157619,0.154868,0.230891,0.160625,0.333414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF703,0.222604,0.101630,0.128992,0.224850,0.157619,0.175506,0.071109,0.230201,0.185125,0.284584,...,0.141732,0.190102,0.101912,0.068699,0.103096,1.000000,0.058706,0.178124,0.050863,0.194664
ZNRD1,0.069420,0.123893,0.187949,0.086748,0.154868,0.054900,0.418163,0.054293,0.126994,0.045645,...,0.055761,0.058149,0.244632,0.189314,0.128546,0.058706,1.000000,0.126709,0.250182,0.083467
ZP3,0.076629,0.361435,0.200042,0.064751,0.230891,0.211739,0.172942,0.119484,0.133095,0.218610,...,0.155753,0.153724,0.125526,0.079986,0.181292,0.178124,0.126709,1.000000,0.159002,0.197240
ZSCAN18,0.097455,0.203343,0.174078,0.059511,0.160625,0.085757,0.245629,0.075171,0.062191,0.103212,...,0.188774,0.133152,0.415823,0.193341,0.380405,0.050863,0.250182,0.159002,1.000000,0.135090


In [38]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.14953670970919472
Median: 0.1275006895525907


# NSC to SMILES

In [39]:
convert = dict(
    pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
        ["NSC", "SMILES"]
    ].values
)
SMILES = [convert[i] for i in drug_response.index]
SMILES

['CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)NC(CCC(=O)O)C(=O)O',
 'C1=NC2=C(N1)C(=S)N=C(N2)N',
 'C1=NC2=C(N1)C(=S)N=CN2',
 'CC(=O)NC1CCC2=CC(=C(C(=C2C3=CC=C(C(=O)C=C13)OC)OC)OC)OC',
 'CN(CCCl)CCCl.Cl',
 'C1=NNC2=C1C(=O)NC=N2',
 'C1(=NC(=NN1)N)N',
 'CC1C(C(=O)NC(C(=O)N2CCCC2C(=O)N(CC(=O)N(C(C(=O)O1)C(C)C)C)C)C(C)C)NC(=O)C3=C4C(=C(C=C3)C)OC5=C(C(=O)C(=C(C5=N4)C(=O)NC6C(OC(=O)C(N(C(=O)CN(C(=O)C7CCCN7C(=O)C(NC6=O)C(C)C)C)C)C(C)C)C)N)C',
 'CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl',
 'C1=CC(=CC=C1CCCC(=O)O)N(CCCl)CCCl',
 'C1CN1P(=S)(N2CC2)N3CC3',
 'C1=CC(=CC=C1CC(C(=O)O)N)N(CCCl)CCCl.Cl',
 'C1CN1C2=NC(=NC(=N2)N3CC3)N4CC4',
 'CCC(=O)OC1CCC2C1(CCC3C2CCC4C3(CC(C(=O)C4)C)C)C',
 'CCN(CC)CCCC(C)NC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)Cl)OC.Cl',
 'C1=C(C(=O)NC(=O)N1)F',
 'CC12CCC3C(C1CCC2OC(=O)CCC4=CC=CC=C4)CCC5=CC(=O)CCC35',
 'CC12CCC3C(C1CCC(=O)O2)CCC4=CC(=O)C=CC34C',
 'CC1C(C(CC(O1)OC2CC(OC(C2O)C)OC3=CC4=CC5=C(C(=O)C(C(C5)C(C(=O)C(C(C)O)O)OC)OC6CC(C(C(O6)C)O)OC7CC(C(C(O7)C)O)OC8CC(C(C(O8)C)O)(C)O)C(=

In [40]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [41]:
mfp = get_morgan_fingerprint(SMILES)
mfp = pd.DataFrame(mfp, index=drug_response.index)

In [42]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim

,740,752,755,757,762,1390,1895,3053,3061,3088,...,808790,808792,809693,810341,810717,811429,812926,812927,813488,820919
740,1.000000,0.685936,0.685936,0.574218,0.699994,0.685936,0.709376,0.445523,0.704684,0.709376,...,0.509650,0.564967,0.560345,0.537268,0.537268,0.523449,0.597385,0.551107,0.551107,0.551107
752,0.685936,1.000000,0.932532,0.685936,0.813195,0.855989,0.870295,0.528053,0.789502,0.747000,...,0.602025,0.657884,0.625260,0.639228,0.648551,0.597385,0.653216,0.643888,0.643888,0.615959
755,0.685936,0.932532,1.000000,0.714071,0.841703,0.884622,0.860755,0.537268,0.798972,0.775314,...,0.629913,0.676576,0.634569,0.667226,0.667226,0.625260,0.662554,0.671900,0.662554,0.634569
757,0.574218,0.685936,0.714071,1.000000,0.728170,0.714071,0.709376,0.472953,0.676576,0.681255,...,0.564967,0.611312,0.588111,0.564967,0.620608,0.588111,0.597385,0.588111,0.606667,0.560345
762,0.699994,0.813195,0.841703,0.728170,1.000000,0.822688,0.865524,0.541879,0.822688,0.855989,...,0.625260,0.671900,0.629913,0.653216,0.653216,0.592747,0.648551,0.667226,0.657884,0.629913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811429,0.523449,0.597385,0.625260,0.588111,0.592747,0.625260,0.629913,0.395444,0.569591,0.574218,...,0.541879,0.551107,0.472953,0.523449,0.523449,1.000000,0.509650,0.564967,0.500463,0.500463
812926,0.597385,0.653216,0.662554,0.597385,0.648551,0.653216,0.676576,0.450089,0.662554,0.620608,...,0.523449,0.625260,0.546492,0.588111,0.569591,0.509650,1.000000,0.555725,0.667226,0.564967
812927,0.551107,0.643888,0.671900,0.588111,0.667226,0.690620,0.667226,0.440959,0.615959,0.611312,...,0.551107,0.569591,0.509650,0.541879,0.541879,0.564967,0.555725,1.000000,0.574218,0.518847
813488,0.551107,0.643888,0.662554,0.606667,0.657884,0.662554,0.676576,0.431838,0.634569,0.620608,...,0.514248,0.597385,0.555725,0.625260,0.514248,0.500463,0.667226,0.574218,1.000000,0.500463


In [43]:
os.makedirs("../nci_data/drug_sim", exist_ok=True)
chunks = np.array_split(drug_sim, 25)

for i, chunk in tqdm(enumerate(chunks)):
    chunk.to_parquet(
        f"../nci_data/drug_sim/drug_sim_part_{i}.parquet", compression="gzip"
    )

/Users/inouey2/base/lib/python3.11/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
25it [00:02,  9.71it/s]


In [44]:
# # How to read
# def natural_sort_key(s):
#     return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', s)]

# file_paths = glob.glob("../nci_data/drug_sim/drug_sim_part_*.parquet")
# sorted_file_paths = sorted(file_paths, key=natural_sort_key)

# drug_sim_reconstructed = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])

In [45]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.5783090139257747
Median: 0.5834776271411233


# Unified Graph

In [46]:
A_dg

,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
740,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
752,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
755,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
757,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
762,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811429,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
812926,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
812927,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
813488,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [47]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,740,752,755,757,762,1390,1895,3053,3061,3088,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
740,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
752,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
755,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
757,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
762,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF703,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNRD1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZSCAN18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [49]:
np.save(
    "../nci_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

In [50]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

In [51]:
os.makedirs("../nci_data/edge_idxs", exist_ok=True)
chunk_size = 3000000

num_chunks_index = math.ceil(edge_index.shape[1] / chunk_size)
for i in range(num_chunks_index):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, edge_index.shape[1])
    chunk = edge_index[:, start:end]
    np.save(f"../nci_data/edge_idxs/edge_idxs_chunk_{i}.npy", chunk)

In [52]:
os.makedirs("../nci_data/edge_attrs", exist_ok=True)
chunk_size = 6000000

num_chunks_attr = math.ceil(edge_attr.shape[0] / chunk_size)
for i in range(num_chunks_attr):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, edge_attr.shape[0])
    chunk = edge_attr[start:end]
    np.save(f"../nci_data/edge_attrs/edge_attr_chunk_{i}.npy", chunk)

In [53]:
# # How to read
# def load_and_combine_chunks(pattern, axis=0):
#     chunk_files = sorted(
#         glob.glob(pattern), key=lambda x: int(x.split("_")[-1].split(".")[0])
#     )

#     chunks = [np.load(f) for f in chunk_files]
#     return np.concatenate(chunks, axis=axis)


# edge_index = load_and_combine_chunks("../nci_data/edge_idxs/*.npy", axis=1)
# edge_attr = load_and_combine_chunks("../nci_data/edge_attrs/*.npy", axis=0)